In [3]:
import numpy as np
import anndata as ad
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

import smoothie
smoothie.suppress_warnings()

In [4]:
#read in raw bin1 and bin50 data

adata_bin1 = ad.read_h5ad("../data/GSM9629357_CS23_E2S1_bin1.h5ad")
adata_bin50 = ad.read_h5ad("../data/GSM9629357_CS23_E2S1_bin50.h5ad")

print(adata_bin1.shape)
print(adata_bin50.shape)

(508126689, 49848)
(662769, 49848)


In [6]:
#filtering and log normalization for bin1 data, same as mouse filters

sc.pp.filter_cells(adata_bin1, min_counts=1)
sc.pp.filter_genes(adata_bin1, min_counts=100)
sc.pp.filter_genes(adata_bin1, min_cells=10)

sc.pp.log1p(adata_bin1)

print(adata_bin1.shape)


(508126689, 30100)


In [7]:
#filtering for bin50 data, only filter sparse genes as already pre-processed

sc.pp.filter_genes(adata_bin50, min_cells=50)


print(adata_bin50.shape)

(662769, 32266)


In [8]:
#smoothing for bin1 data

target_microns = 20.0
micron_to_unit_conversion = 2
sm_adata_bin1 = smoothie.run_parallelized_smoothing(
    adata_bin1,
    grid_based_or_not=True,
    gaussian_sd=target_microns * micron_to_unit_conversion,
    min_spots_under_gaussian=1000
)

#save smoothed data to data folder
sm_adata_bin1.write_h5ad("../data/GSM9629357_CS23_E2S1_bin1_smooth.h5ad")

Checking dataset point density...
Fitting grid to tissue coordinates...
Variation in X-direction: 49527.0
Variation in Y-direction: 70774.0
Number of points in fitted grid: 1490669
Gaussian smoothing will run in 36 chunks.
Smoothing chunk: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 
Total runtime for grid-based Gaussian smoothing: 2261.77 seconds.



In [9]:
#smoothing for bin50 data

target_microns = 30.0
micron_to_unit_conversion = 2
sm_adata_bin50 = smoothie.run_parallelized_smoothing(
    adata_bin50,
    grid_based_or_not=False,
    gaussian_sd=target_microns * micron_to_unit_conversion,
    min_spots_under_gaussian=5
)

#save smoothed data to data folder
sm_adata_bin50.write_h5ad("../data/GSM9629357_CS23_E2S1_bin50_smooth.h5ad")

Storage for smoothed count matrix will be 85.54 GB.
Checking dataset point density...
Gaussian smoothing will run in 16 chunks.
Smoothing chunk: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
Total runtime for in-place Gaussian smoothing: 768.89 seconds.



In [ ]:
#compute correlation matrix for bin 1

pearsonR_mat_bin1, pval_mat_bin1 = smoothie.compute_correlation_matrix(sm_adata_bin1.X)

#save matrices to data folder
np.save(f"../data/GSM9629357_CS23_E2S1_bin1_pearsonR.npy", pearsonR_mat_bin1)
np.save(f"../data/GSM9629357_CS23_E2S1_bin1_pval.npy", pval_mat_bin1)
np.save(f"../data/GSM9629357_CS23_E2S1_bin1_gene_names.npy", sm_adata_bin1.var_names.to_numpy())

compute_correlation_matrix running...


In [ ]:
#compute correlation matrix for bin 50

pearsonR_mat_bin50, pval_mat_bin50 = smoothie.compute_correlation_matrix(sm_adata_bin50.X)

#save matrices to data folder
np.save(f"../data/GSM9629357_CS23_E2S1_bin50_pearsonR.npy", pearsonR_mat_bin50)
np.save(f"../data/GSM9629357_CS23_E2S1_bin50_pval.npy", pval_mat_bin50)
np.save(f"../data/GSM9629357_CS23_E2S1_bin50_gene_names.npy", sm_adata_bin50.var_names.to_numpy())

In [ ]:
#create graphs to determine clustering parameters for bin1

smoothie.select_clustering_params(
    gene_names=sm_adata_bin1.var_names,
    pearsonR_mat=pearsonR_mat_bin1,
    output_folder=None,
    pcc_cutoffs=[0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    clustering_powers=[1, 3, 5, 7, 9],
    min_genes_for_module=5
)

In [ ]:
#create graphs to determine clustering parameters for bin50

smoothie.select_clustering_params(
    gene_names=sm_adata_bin50.var_names,
    pearsonR_mat=pearsonR_mat_bin50,
    output_folder=None,
    pcc_cutoffs=[0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    clustering_powers=[1, 3, 5, 7, 9],
    min_genes_for_module=5
)

In [ ]:
#construct network and save to Cytoscape folder for bin1

edge_list_bin1, node_label_df_bin1 = smoothie.make_spatial_network(
    pearsonR_mat=pearsonR_mat_bin1, # don't change
    gene_names=sm_adata_bin1.var_names, # don't change
    pcc_cutoff=0.4,
    clustering_power=3,
    output_folder='../Cytoscape/bin1'
)

# Filter node_label_df for only genes within modules of size 2 or more.
modules_df_bin1 = node_label_df_bin1.groupby('module_label').filter(lambda x: len(x) >= 2)

# Examine modules
modules_df_bin1

In [ ]:
#construct network and save to Cytoscape folder for bin50

edge_list_bin50, node_label_df_bin50 = smoothie.make_spatial_network(
    pearsonR_mat=pearsonR_mat_bin50, # don't change
    gene_names=sm_adata_bin50.var_names, # don't change
    pcc_cutoff=0.4,
    clustering_power=3,
    output_folder='../Cytoscape/bin50'
)

# Filter node_label_df for only genes within modules of size 2 or more.
modules_df_bin50 = node_label_df_bin50.groupby('module_label').filter(lambda x: len(x) >= 2)

# Examine modules
modules_df_bin50

In [ ]:
#plot the gene modules for bin1

smoothie.plot_modules(
    sm_adata_bin1,
    node_label_df_bin1,
    output_folder='./module_plots',
    min_genes=3, # Minimum number of genes in a module to plot the module
    spot_size=50, # Adjust for optimal visualization
    plots_per_row=10,
    dpi=150 # low resolution here for github, better to choose 300-600!
)

In [ ]:
#plot the gene modules for bin50

smoothie.plot_modules(
    sm_adata_bin50,
    node_label_df_bin50,
    output_folder='./module_plots',
    min_genes=3, # Minimum number of genes in a module to plot the module
    spot_size=50, # Adjust for optimal visualization
    plots_per_row=10,
    dpi=150 # low resolution here for github, better to choose 300-600!
)